# 02 — Propriétés, méthodes de classe et méthodes statiques

## Objectifs pédagogiques

À la fin de ce notebook, vous saurez :

- exposer un attribut contrôlé avec `@property`
- écrire un setter `@x.setter` avec validation
- mémoïser un calcul coûteux avec `@cached_property`
- distinguer `@classmethod` et `@staticmethod`
- écrire des constructeurs alternatifs `from_*` avec `@classmethod`

## Prérequis — ce que vous connaissez déjà

À ce stade de la formation intermédiaire, vous maîtrisez :

- classes, `__init__`, `self` (notebook 01)
- attributs d'instance et de classe
- `__repr__` / `__str__`
- décorateurs de base (à travers `@property`, on en voit l'usage)
- exceptions (`ValueError`)

Ce que nous n'avons **pas encore vu** (et que nous n'utiliserons donc pas dans ce notebook) :

- l'héritage et `super()` (notebook 03)
- `__eq__`, `__lt__`, surcharge arithmétique (notebook 04)
- `ABC` et `Protocol` (notebook 05)
- écrire ses propres décorateurs (jour 3)

## Plan

1. Pourquoi `@property` : accesseur sans parenthèses
2. Setter et validation
3. Propriété en lecture seule
4. `@cached_property` : calcul coûteux mémorisé
5. `@classmethod` : méthode qui reçoit la classe
6. Constructeurs alternatifs `from_*`
7. `@staticmethod` : fonction logée dans la classe
8. Quand choisir quoi (tableau de décision)
9. Synthèse
10. Exercices

---

## 1. Pourquoi `@property` : accesseur sans parenthèses

Partons du notebook 01. On avait une méthode `capacite()` pour contrôler l'accès :

In [ ]:
class Salle:
    def __init__(self, nom: str, capacite: int) -> None:
        self.nom = nom
        self._capacite = capacite

    def capacite(self) -> int:
        return self._capacite


In [ ]:
mars = Salle("Mars", 12)


In [ ]:
mars.capacite()  # parenthèses obligatoires


Problème : l'utilisateur de la classe doit se souvenir que c'est une méthode, pas un attribut. Si plus tard on veut simplifier et exposer directement `self._capacite`, **tout le code appelant casse** (il faudrait enlever les parenthèses partout).

`@property` résout ça : on écrit une **méthode**, mais le code appelant y accède **sans parenthèses**, comme à un attribut.

In [ ]:
class Salle:
    def __init__(self, nom: str, capacite: int) -> None:
        self.nom = nom
        self._capacite = capacite

    @property
    def capacite(self) -> int:
        return self._capacite


In [ ]:
mars = Salle("Mars", 12)


In [ ]:
mars.capacite  # sans parenthèses !


La règle d'or est simple : **on expose des attributs, jamais des méthodes getter/setter**. Si un jour le calcul devient plus riche, `@property` permet de migrer sans casser les appelants.

---

## 2. Setter et validation

Par défaut, une `@property` est **en lecture seule** : tenter de l'assigner lève `AttributeError`.

In [ ]:
try:
    mars.capacite = 20
except AttributeError as exc:
    print('erreur :', exc)


Pour autoriser l'écriture, on définit un setter avec `@<nom>.setter`. C'est l'endroit **idéal** pour valider la nouvelle valeur.

In [ ]:
class Salle:
    def __init__(self, nom: str, capacite: int) -> None:
        self.nom = nom
        self.capacite = capacite  # passe par le setter → validation immédiate

    @property
    def capacite(self) -> int:
        return self._capacite

    @capacite.setter
    def capacite(self, valeur: int) -> None:
        if not isinstance(valeur, int) or valeur <= 0:
            raise ValueError(f'capacité invalide : {valeur!r}')
        self._capacite = valeur


In [ ]:
mars = Salle("Mars", 12)


In [ ]:
mars.capacite = 20


In [ ]:
mars.capacite


In [ ]:
try:
    mars.capacite = -5
except ValueError as exc:
    print(exc)


### Passer par le setter depuis `__init__`

Notez que dans `__init__` on écrit `self.capacite = capacite` (pas `self._capacite`). Ça déclenche le setter et donc la validation **au moment de la création**. Un objet invalide ne peut pas exister.

---

## 3. Propriété en lecture seule — calcul dérivé

Une `@property` sans setter sert aussi à exposer une valeur **dérivée** des autres attributs, sans dupliquer d'état.

In [ ]:
class Rectangle:
    def __init__(self, largeur: float, hauteur: float) -> None:
        self.largeur = largeur
        self.hauteur = hauteur

    @property
    def aire(self) -> float:
        return self.largeur * self.hauteur

    @property
    def perimetre(self) -> float:
        return 2 * (self.largeur + self.hauteur)


In [ ]:
r = Rectangle(4, 3)


In [ ]:
r.aire, r.perimetre


In [ ]:
r.largeur = 5


In [ ]:
r.aire  # recalculé automatiquement


**Règle** : ne stockez jamais une valeur qui peut être dérivée des autres. Ça évite les états incohérents (un rectangle dont l'aire mentirait sur sa largeur).

---

## 4. `@cached_property` : calcul coûteux mémorisé

Tiré de `functools` : une `@cached_property` est évaluée **à la première lecture**, puis mémorisée sur l'instance. Pratique pour un calcul lourd.

In [ ]:
import time
from functools import cached_property


class Rapport:
    def __init__(self, donnees: list[int]) -> None:
        self.donnees = donnees

    @cached_property
    def resume(self) -> dict[str, float]:
        print('… calcul du résumé …')
        time.sleep(0.3)  # simule un calcul lourd
        n = len(self.donnees)
        return {
            'n': n,
            'moyenne': sum(self.donnees) / n,
            'max': max(self.donnees),
            'min': min(self.donnees),
        }


In [ ]:
r = Rapport([1, 2, 3, 4, 5])


In [ ]:
r.resume  # premier appel → message affiché


In [ ]:
r.resume  # appels suivants → instantané, pas de message


### Différence avec `@property`

- `@property` → **recalculé à chaque accès** ; idéal pour un calcul dérivé léger.
- `@cached_property` → **calculé une seule fois** par instance ; idéal pour un calcul coûteux et stable.

⚠️ `@cached_property` **stocke** le résultat dans `self.__dict__`. Si `self.donnees` change, le cache ne se rafraîchit pas automatiquement. Il faudra `del r.resume` pour le réinitialiser.

In [ ]:
r.donnees.append(100)
r.resume  # ← PAS à jour, toujours l'ancien max=5


In [ ]:
del r.resume  # invalide le cache
r.resume      # recalcul


---

## 5. `@classmethod` : méthode qui reçoit la classe

Une `@classmethod` reçoit automatiquement la **classe** en premier argument, conventionnellement nommé `cls`. Utile pour :

- écrire des **constructeurs alternatifs** (`from_csv`, `from_dict`, `from_json`) ;
- accéder à des attributs de classe sans instance.

In [ ]:
class Salle:
    tva: float = 0.20  # attribut de classe

    def __init__(self, nom: str, capacite: int, prix_ht: float) -> None:
        self.nom = nom
        self.capacite = capacite
        self.prix_ht = prix_ht

    @classmethod
    def tva_actuelle(cls) -> float:
        return cls.tva

    @property
    def prix_ttc(self) -> float:
        return self.prix_ht * (1 + type(self).tva)


In [ ]:
Salle.tva_actuelle()


In [ ]:
mars = Salle("Mars", 12, 100.0)


In [ ]:
round(mars.prix_ttc, 2)


---

## 6. Constructeurs alternatifs `from_*`

L'usage le plus fréquent de `@classmethod` en Python.

In [ ]:
class Salle:
    def __init__(self, nom: str, capacite: int) -> None:
        self.nom = nom
        self.capacite = capacite

    def __repr__(self) -> str:
        return f'Salle({self.nom!r}, {self.capacite})'

    @classmethod
    def from_dict(cls, data: dict[str, object]) -> 'Salle':
        return cls(nom=str(data['nom']), capacite=int(data['capacite']))

    @classmethod
    def from_csv_row(cls, ligne: str) -> 'Salle':
        nom, cap = ligne.strip().split(',')
        return cls(nom, int(cap))


In [ ]:
Salle.from_dict({"nom": "Mars", "capacite": 12})


In [ ]:
Salle.from_csv_row("Venus,6")


### Pourquoi `cls(...)` et pas `Salle(...)` dans le code ?

Parce que si demain une sous-classe `SalleVIP` hérite, son `SalleVIP.from_dict(...)` renverra automatiquement une `SalleVIP`, pas une `Salle`. C'est l'intérêt principal de `@classmethod` par rapport à une simple fonction libre.

---

## 7. `@staticmethod` : fonction logée dans la classe

Une `@staticmethod` ne reçoit **ni instance ni classe**. C'est essentiellement une fonction libre qu'on range dans la classe par cohérence de namespace. Utile pour des helpers liés au domaine de la classe.

In [ ]:
class Salle:
    def __init__(self, nom: str, capacite: int) -> None:
        self.nom = nom
        self.capacite = capacite

    @staticmethod
    def creneau_valide(creneau: str) -> bool:
        jours = ('lundi', 'mardi', 'mercredi', 'jeudi', 'vendredi')
        return any(creneau.startswith(j) for j in jours)


In [ ]:
Salle.creneau_valide("lundi 9h")


In [ ]:
Salle.creneau_valide("samedi 9h")


En pratique, beaucoup de gens préfèrent mettre ces helpers dans un **module** à côté plutôt que comme `@staticmethod`. Les deux sont acceptables ; on préfère `@staticmethod` quand le helper est **fortement lié** au domaine de la classe.

---

## 8. Quand choisir quoi

Tableau de décision à afficher au-dessus de l'écran.

| Ce que je veux | Outil |
|---|---|
| Accéder à une valeur calculée à la lecture | `@property` |
| Contrôler/valider une écriture | `@property` + `@x.setter` |
| Calcul lourd, stable pour la durée de vie de l'objet | `@cached_property` |
| Constructeur alternatif (`from_dict`, `from_csv`…) | `@classmethod` |
| Accéder à un attribut de classe indépendamment de l'instance | `@classmethod` |
| Helper lié au domaine mais sans `self` ni `cls` | `@staticmethod` |
| Helper technique sans lien fort avec la classe | fonction libre dans un module |


---

## Synthèse

| Décorateur | Premier arg | Usage principal |
|---|---|---|
| `@property` | `self` | Accesseur en lecture, validation en écriture |
| `@cached_property` | `self` | Calcul coûteux mémorisé par instance |
| `@classmethod` | `cls` | Constructeurs alternatifs, accès aux attributs de classe |
| `@staticmethod` | *(aucun)* | Fonction liée à la classe mais indépendante |


### Règles à retenir

1. **Exposez des attributs, pas des getters**. Si un contrôle devient nécessaire, `@property` permet de migrer sans casser les appelants.
2. **Valider dans le setter**, et appeler le setter depuis `__init__`. Un objet invalide ne doit pas pouvoir exister.
3. **`@cached_property` n'invalide jamais son cache automatiquement** : utilisez-la sur des objets dont l'état source ne change pas.
4. **`@classmethod` pour les `from_xxx`** : c'est l'idiome, et c'est compatible avec l'héritage.
5. **`@staticmethod` est rare** : quand vous en avez envie, demandez-vous d'abord si une fonction libre dans le module ne ferait pas mieux.

---

## Exercices

Les exercices sont gradués. Tous utilisent des fonctions typées (PEP 604).

### Exercice 1 — Propriété `plein_nom` *(facile)*

Écrire une classe `Utilisateur` avec `prenom: str` et `nom: str` en attributs, et une `@property` `plein_nom` qui renvoie `'Prenom Nom'`.

In [ ]:
# Votre code ici


In [ ]:
# ▶ Une fois votre solution écrite ci-dessus, exécutez cette cellule
# pour signaler à votre formateur que vous avez tenté l'exercice.
import sys
from pathlib import Path
for _p in (Path.cwd(), *Path.cwd().parents):
    if (_p / "_common" / "utils_pedagogie.py").exists():
        sys.path.insert(0, str(_p / "_common")); break
from utils_pedagogie import marquer_tentative
marquer_tentative(notebook="02_Proprietes", exercice=1)


<details>
<summary>📖 Voir la correction</summary>

```python
class Utilisateur:
    def __init__(self, prenom: str, nom: str) -> None:
        self.prenom = prenom
        self.nom = nom

    @property
    def plein_nom(self) -> str:
        return f'{self.prenom} {self.nom}'


u = Utilisateur('Alice', 'Martin')
print(u.plein_nom)
```

</details>

### Exercice 2 — Setter avec validation *(facile)*

Écrire une classe `Temperature` avec un attribut interne `_celsius`. Exposer une `@property celsius` en lecture/écriture qui refuse les valeurs inférieures à `-273.15` (zéro absolu) en levant `ValueError`.

In [ ]:
# Votre code ici


In [ ]:
# ▶ Une fois votre solution écrite ci-dessus, exécutez cette cellule
# pour signaler à votre formateur que vous avez tenté l'exercice.
import sys
from pathlib import Path
for _p in (Path.cwd(), *Path.cwd().parents):
    if (_p / "_common" / "utils_pedagogie.py").exists():
        sys.path.insert(0, str(_p / "_common")); break
from utils_pedagogie import marquer_tentative
marquer_tentative(notebook="02_Proprietes", exercice=2)


<details>
<summary>📖 Voir la correction</summary>

```python
class Temperature:
    def __init__(self, celsius: float) -> None:
        self.celsius = celsius

    @property
    def celsius(self) -> float:
        return self._celsius

    @celsius.setter
    def celsius(self, valeur: float) -> None:
        if valeur < -273.15:
            raise ValueError('en dessous du zéro absolu')
        self._celsius = float(valeur)


t = Temperature(20.0)
print(t.celsius)
t.celsius = -10.0
try:
    t.celsius = -300.0
except ValueError as exc:
    print(exc)
```

</details>

### Exercice 3 — Conversion via propriété *(moyen)*

Étendre la classe `Temperature` précédente : ajouter une propriété `fahrenheit` qui expose la même valeur en Fahrenheit. Écrire :

- le **getter** `fahrenheit` calculé à partir de `celsius` ;
- le **setter** `fahrenheit` qui reçoit une valeur Fahrenheit et met à jour `_celsius` (en validant au passage).

L'objet doit rester **cohérent** : lire `celsius` et `fahrenheit` donne toujours deux vues de la même température.

In [ ]:
# Votre code ici


In [ ]:
# ▶ Une fois votre solution écrite ci-dessus, exécutez cette cellule
# pour signaler à votre formateur que vous avez tenté l'exercice.
import sys
from pathlib import Path
for _p in (Path.cwd(), *Path.cwd().parents):
    if (_p / "_common" / "utils_pedagogie.py").exists():
        sys.path.insert(0, str(_p / "_common")); break
from utils_pedagogie import marquer_tentative
marquer_tentative(notebook="02_Proprietes", exercice=3)


<details>
<summary>📖 Voir la correction</summary>

```python
class Temperature:
    def __init__(self, celsius: float) -> None:
        self.celsius = celsius

    @property
    def celsius(self) -> float:
        return self._celsius

    @celsius.setter
    def celsius(self, valeur: float) -> None:
        if valeur < -273.15:
            raise ValueError('en dessous du zéro absolu')
        self._celsius = float(valeur)

    @property
    def fahrenheit(self) -> float:
        return self._celsius * 9 / 5 + 32

    @fahrenheit.setter
    def fahrenheit(self, valeur: float) -> None:
        self.celsius = (valeur - 32) * 5 / 9


t = Temperature(100.0)
print(t.fahrenheit)  # 212
t.fahrenheit = 32.0
print(t.celsius)     # 0.0
```

</details>

### Exercice 4 — `@classmethod` — constructeurs depuis JSON et CSV *(moyen)*

Reprendre la classe `Reservation` (fil rouge) avec `salle`, `creneau`, `organisateur`. Ajouter :

- `Reservation.from_dict(data)` qui attend `{'salle': ..., 'creneau': ..., 'organisateur': ...}` ;
- `Reservation.from_csv_row(row)` qui prend une chaîne `'Mars;lundi 9h;Alice'` (séparateur `;`).

Tester les deux constructeurs.

In [ ]:
# Votre code ici


In [ ]:
# ▶ Une fois votre solution écrite ci-dessus, exécutez cette cellule
# pour signaler à votre formateur que vous avez tenté l'exercice.
import sys
from pathlib import Path
for _p in (Path.cwd(), *Path.cwd().parents):
    if (_p / "_common" / "utils_pedagogie.py").exists():
        sys.path.insert(0, str(_p / "_common")); break
from utils_pedagogie import marquer_tentative
marquer_tentative(notebook="02_Proprietes", exercice=4)


<details>
<summary>📖 Voir la correction</summary>

```python
class Reservation:
    def __init__(self, salle: str, creneau: str, organisateur: str) -> None:
        self.salle = salle
        self.creneau = creneau
        self.organisateur = organisateur

    def __repr__(self) -> str:
        return (
            f'Reservation({self.salle!r}, {self.creneau!r}, {self.organisateur!r})'
        )

    @classmethod
    def from_dict(cls, data: dict[str, str]) -> 'Reservation':
        return cls(data['salle'], data['creneau'], data['organisateur'])

    @classmethod
    def from_csv_row(cls, row: str) -> 'Reservation':
        salle, creneau, org = row.strip().split(';')
        return cls(salle, creneau, org)


print(Reservation.from_dict({'salle': 'Mars', 'creneau': 'lundi 9h', 'organisateur': 'Alice'}))
print(Reservation.from_csv_row('Venus;mardi 14h;Bob'))
```

</details>

### Exercice 5 — `@cached_property` — statistiques figées *(moyen)*

Écrire une classe `Mesures` qui reçoit une `list[float]` et expose trois `@cached_property` : `moyenne`, `min_`, `max_`. Ajouter une méthode `invalide()` qui supprime les trois caches (via `del`) pour forcer un recalcul au prochain accès.

In [ ]:
# Votre code ici


In [ ]:
# ▶ Une fois votre solution écrite ci-dessus, exécutez cette cellule
# pour signaler à votre formateur que vous avez tenté l'exercice.
import sys
from pathlib import Path
for _p in (Path.cwd(), *Path.cwd().parents):
    if (_p / "_common" / "utils_pedagogie.py").exists():
        sys.path.insert(0, str(_p / "_common")); break
from utils_pedagogie import marquer_tentative
marquer_tentative(notebook="02_Proprietes", exercice=5)


<details>
<summary>📖 Voir la correction</summary>

```python
from functools import cached_property


class Mesures:
    def __init__(self, valeurs: list[float]) -> None:
        self.valeurs = valeurs

    @cached_property
    def moyenne(self) -> float:
        return sum(self.valeurs) / len(self.valeurs)

    @cached_property
    def min_(self) -> float:
        return min(self.valeurs)

    @cached_property
    def max_(self) -> float:
        return max(self.valeurs)

    def invalide(self) -> None:
        for nom in ('moyenne', 'min_', 'max_'):
            self.__dict__.pop(nom, None)


m = Mesures([1.0, 2.0, 3.0, 4.0])
print(m.moyenne, m.min_, m.max_)
m.valeurs.append(100.0)
m.invalide()
print(m.moyenne, m.min_, m.max_)
```

</details>

### Exercice 6 — Propriété dérivée + validation couplée *(difficile)*

Écrire une classe `Intervalle` avec deux propriétés `debut` et `fin` (en `int`, minutes depuis minuit). Règles :

- à toute lecture, `duree` doit renvoyer `fin - debut` ;
- le setter de `debut` doit refuser une valeur `>= fin` ;
- le setter de `fin` doit refuser une valeur `<= debut` ;
- sur `__init__`, vérifier `0 <= debut < fin <= 24*60`.

Valider tout ça à l'aide de quelques assertions et de quelques `try`/`except`.

In [ ]:
# Votre code ici


In [ ]:
# ▶ Une fois votre solution écrite ci-dessus, exécutez cette cellule
# pour signaler à votre formateur que vous avez tenté l'exercice.
import sys
from pathlib import Path
for _p in (Path.cwd(), *Path.cwd().parents):
    if (_p / "_common" / "utils_pedagogie.py").exists():
        sys.path.insert(0, str(_p / "_common")); break
from utils_pedagogie import marquer_tentative
marquer_tentative(notebook="02_Proprietes", exercice=6)


<details>
<summary>📖 Voir la correction</summary>

```python
class Intervalle:
    def __init__(self, debut: int, fin: int) -> None:
        if not (0 <= debut < fin <= 24 * 60):
            raise ValueError('intervalle invalide')
        self._debut = debut
        self._fin = fin

    @property
    def debut(self) -> int:
        return self._debut

    @debut.setter
    def debut(self, valeur: int) -> None:
        if valeur >= self._fin:
            raise ValueError('debut >= fin')
        self._debut = valeur

    @property
    def fin(self) -> int:
        return self._fin

    @fin.setter
    def fin(self, valeur: int) -> None:
        if valeur <= self._debut:
            raise ValueError('fin <= debut')
        self._fin = valeur

    @property
    def duree(self) -> int:
        return self._fin - self._debut


iv = Intervalle(9 * 60, 10 * 60)
assert iv.duree == 60
iv.fin = 11 * 60
assert iv.duree == 120
try:
    iv.debut = 12 * 60
except ValueError as exc:
    print(exc)
```

</details>

---

## Ressources externes

### Documentation officielle
- [`property` — stdlib](https://docs.python.org/3/library/functions.html#property)
- [`functools.cached_property`](https://docs.python.org/3/library/functools.html#functools.cached_property)
- [Descriptor HowTo](https://docs.python.org/3/howto/descriptor.html) *(pour comprendre ce qui se passe sous `@property`)*

### PEPs de référence
- **PEP 252** — *Making Types Look More Like Classes*

### Lectures complémentaires
- Fluent Python (Ramalho), chap. 22 *Dynamic Attributes and Properties*.